In [ ]:
#loading required packages
import re
import numpy as np
import tweepy 
from tweepy import OAuthHandler 
from textblob import TextBlob 
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from wordcloud import WordCloud
from better_profanity import profanity
import seaborn
import sklearn
from sklearn.metrics import precision_score,recall_score,accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_recall_fscore_support
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from transformers import TFBertModel, BertTokenizer
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from tensorflow.keras.layers import Input, Embedding, Conv1D, GlobalMaxPooling1D, LSTM, Dense
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

main_path = r'C:\Users\Jerry\Desktop\Xacking PhD\dataset\covid'

In [ ]:
#reading the dataset
# df = pd.read_csv('ebola_tweets.csv')
df = pd.read_excel(main_path + '/covid 19 clinical dataset.xlsx')

In [ ]:
df

In [ ]:
# Remove the 'Patient ID' column
df = df.drop(columns=['Patient ID'])

In [ ]:
df.rename(columns={"SARS-Cov-2 exam result": "labels"}, inplace=True)
df.head()

In [ ]:
df["labels"].value_counts()

In [ ]:
# Identify columns where all values are 0 or NaN
cols_all_zero_or_nan = [
    col for col in df.columns
    if df[col].replace(0, np.nan).isna().all()
]

print("Columns with all 0 or NaN:", cols_all_zero_or_nan)

In [ ]:
df = df.drop(columns=cols_all_zero_or_nan)

In [ ]:
df.head()

In [ ]:
#label encoding
mapping = {'positive':1,'negative':0}
df['labels'] = df['labels'].map(mapping)

In [ ]:
df["labels"].value_counts()

In [ ]:
df.head()

In [ ]:
# Select columns that are numeric
numeric_cols = df.select_dtypes(include=[np.number]).columns

# Columns that are object/string type
non_numeric_cols = df.select_dtypes(include=['object']).columns
print("Non-numeric columns:", non_numeric_cols)

In [ ]:
print("Numeric columns:", numeric_cols)

In [ ]:
# Identify categorical columns
categorical_cols = df.select_dtypes(include=['object', 'category']).columns
categorical_cols

In [ ]:
# create a copy of df:
df2 = df.copy()

In [ ]:
# fill all NANs in categorical columns with 0
df2[categorical_cols] = df2[categorical_cols].fillna(0)

In [ ]:
# encode all categorical columns (assume all columns are ordinal)
for col in categorical_cols:
    le = LabelEncoder()
    df2[col] = le.fit_transform(df2[col].astype(str))

In [ ]:
# 0 is for NAN
df["Parainfluenza 3"].value_counts(), df2["Parainfluenza 3"].value_counts()

In [ ]:
# 0 is for NAN
df["Urine - Yeasts"].value_counts(), df2["Urine - Yeasts"].value_counts()

In [ ]:
# fill all NANs in numerical columns with median
df2[numeric_cols] = df2[numeric_cols].fillna(df2[numeric_cols].median())
df2.head()

In [ ]:
# check if any NAN exist in df2
df2.isna().any().any()

In [ ]:
import pandas as pd

# Separate classes
df_neg = df2[df2['labels'] == 0]
df_pos = df2[df2['labels'] == 1]

# Find minority class size
n_samples = min(len(df_neg), len(df_pos))

# Sample equally
df_neg_sampled = df_neg.sample(n=n_samples, random_state=42)
df_pos_sampled = df_pos.sample(n=n_samples, random_state=42)

# Combine and shuffle
df_balanced = pd.concat([df_neg_sampled, df_pos_sampled]) \
                 .sample(frac=1, random_state=42) \
                 .reset_index(drop=True)

# Check balance
print(df_balanced['labels'].value_counts())


In [ ]:
# separate features from label and scale the features
X = df2.drop(columns=['labels'])
labels = df2['labels'].values

# scaler = StandardScaler()
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

In [ ]:
# reshape scaled features (X) for CNN and LSTM ie from (num_samples, _num_features) to 
# (sample, timesteps, channels)
X_seq = X_scaled.reshape(X_scaled.shape[0], X_scaled.shape[1], 1)

In [ ]:
X_seq.shape, labels.shape

In [ ]:
# convert X back to DataFrame
X_names = X.columns
X_df = pd.DataFrame(X_scaled, columns=X_names)
X_df.shape

In [ ]:
# # Extract CNN features
# from tensorflow.keras.layers import Input, Conv1D, GlobalMaxPooling1D, Concatenate
# from tensorflow.keras.models import Model

# inp = Input(shape=(X_seq.shape[1], 1))

# conv3 = Conv1D(64, 3, activation='relu')(inp)
# conv4 = Conv1D(64, 4, activation='relu')(inp)
# conv5 = Conv1D(64, 5, activation='relu')(inp)

# cnn = Concatenate()([
#     GlobalMaxPooling1D()(conv3),
#     GlobalMaxPooling1D()(conv4),
#     GlobalMaxPooling1D()(conv5)
# ])

# cnn_model = Model(inp, cnn)

In [ ]:
from tensorflow.keras.layers import Input, Conv1D, GlobalMaxPooling1D
from tensorflow.keras.models import Model

inp = Input(shape=(X_seq.shape[1], 1))

cnn = Conv1D(32, 3, activation='relu')(inp)
cnn = Conv1D(64, 3, activation='relu')(cnn)
cnn = GlobalMaxPooling1D()(cnn)

cnn_model = Model(inp, cnn)

In [ ]:
# Extract lstm features
from tensorflow.keras.layers import LSTM, Bidirectional

lstm = Bidirectional(
    LSTM(32, return_sequences=False)
)(inp)

lstm_model = Model(inp, lstm)

In [ ]:
cnn_features = cnn_model.predict(X_seq)
lstm_features = lstm_model.predict(X_seq)

In [ ]:
# combine features Including raw features improves stability
import numpy as np

combined_features = np.concatenate(
    [cnn_features, lstm_features, X_scaled],
    axis=1
)

In [ ]:
combined_features.shape, cnn_features.shape, lstm_features.shape, X_scaled.shape

In [ ]:
train_data, test_data, train_labels, test_labels = train_test_split(combined_features, labels, test_size=0.2, random_state=0)

# train_data, val_data, train_labels, val_labels = train_test_split(train_data, train_labels, test_size=0.2, random_state=0)

In [ ]:
# train a RF model
rf = RandomForestClassifier(n_estimators=50, random_state=42)
rf.fit(train_data, train_labels)

In [ ]:
y_pred = rf.predict(test_data)

In [ ]:
print("Classification Report:\n")
print(classification_report(test_labels, y_pred))

In [ ]:
import shap
import matplotlib.pyplot as plt

# Initialize TreeExplainer for Random Forest
# Using interventional mode avoids additivity check errors when slicing features
explainer = shap.TreeExplainer(rf, feature_perturbation='interventional')

In [ ]:
# Compute SHAP values for the test data (full combined features)
shap_values = explainer.shap_values(test_data)

In [ ]:
# Now select only the last X_scaled features for plotting
X_scaled_test = test_data[:, -X_scaled.shape[1]:]  #extract only the clinical features from test data
shap_values_scaled = shap_values[:, -X_scaled.shape[1]:, 1]  # extract clinical features and select positive class

In [ ]:
shap_values_scaled.shape, X_scaled_test.shape, 

In [ ]:
shap.summary_plot(
    shap_values_scaled, 
    X_scaled_test, 
    feature_names=X_names
)

In [ ]:
# ---- Bar plot ----
shap.summary_plot(
    shap_values_scaled, 
    X_scaled_test,
    feature_names=X_names, 
    plot_type="bar"
)

In [ ]:
# ---- Dependence plot example for first feature ----
shap.dependence_plot(
    X_names[0], 
    shap_values_scaled, 
    X_scaled_test,
    feature_names=X_names
)

In [ ]:
# ---- Force plot for a single sample ----
sample_index = 0
shap.initjs()
shap.force_plot(
    explainer.expected_value[1], 
    shap_values_scaled[sample_index], 
    X_scaled_test[sample_index], 
    feature_names=X_names
)

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import numpy as np

# PCA
pca = PCA(n_components=2)
features_2d = pca.fit_transform(combined_features)

labels_np = np.array(labels)

plt.figure(figsize=(7, 6))

# Negative class (0)
idx_neg = labels_np == 0
plt.scatter(
    features_2d[idx_neg, 0],
    features_2d[idx_neg, 1],
    color='blue',
    label='Negative',
    alpha=0.6
)

# Positive class (1)
idx_pos = labels_np == 1
plt.scatter(
    features_2d[idx_pos, 0],
    features_2d[idx_pos, 1],
    color='red',
    label='Positive',
    alpha=0.6
)

plt.xlabel("PCA 1")
plt.ylabel("PCA 2")
plt.title("PCA Projection of CNN-Extracted Tweet Features")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Feature importance
import numpy as np
import matplotlib.pyplot as plt

importances = rf.feature_importances_
colors = plt.cm.viridis(importances / importances.max())

plt.bar(range(len(importances)), importances, color=colors)
plt.xlabel("Feature Index")
plt.ylabel("Importance")
plt.title("Random Forest Feature Importances")
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Number of features
n_cnn = cnn_features.shape[1]
n_lstm = lstm_features.shape[1]

# Feature importances from trained RF
importances = rf.feature_importances_

# Split importances
cnn_importances = importances[:n_cnn]
lstm_importances = importances[n_cnn:n_cnn + n_lstm]

# Colors
cnn_colors = plt.cm.Blues(cnn_importances / cnn_importances.max())
lstm_colors = plt.cm.Oranges(lstm_importances / lstm_importances.max())

# -----------------------------
# Plot side by side
# -----------------------------
plt.figure(figsize=(14, 5))

# CNN feature importance
plt.subplot(1, 2, 1)
plt.bar(range(n_cnn), cnn_importances, color=cnn_colors)
plt.xlabel("CNN Feature Index")
plt.ylabel("Importance")
plt.title("(A) Feature Importance (CNN)")

# LSTM feature importance
plt.subplot(1, 2, 2)
plt.bar(range(n_lstm), lstm_importances, color=lstm_colors)
plt.xlabel("LSTM Feature Index")
plt.ylabel("Importance")
plt.title("(B) Feature Importance (LSTM)")

plt.tight_layout()
plt.show()


In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

cm = confusion_matrix(test_labels, y_pred)

plt.figure(figsize=(6, 5))
class_names = ["Negative", "Positive"]

sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=class_names,
    yticklabels=class_names
)


plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix")
plt.tight_layout()
plt.show()


In [ ]:
# Precision–Recall (PR) Curve 
from sklearn.metrics import precision_recall_curve

y_prob = rf.predict_proba(test_data)[:,1]
precision, recall, _ = precision_recall_curve(test_labels, y_prob)

plt.plot(recall, precision)
plt.xlabel("Recall")
plt.ylabel("Precision")


In [ ]:
# ROC
from sklearn.metrics import roc_curve, auc

fpr, tpr, _ = roc_curve(test_labels, y_prob)
roc_auc = auc(fpr, tpr)

plt.plot(fpr, tpr, label=f"AUC = {roc_auc:.2f}")
plt.legend()


### Model Training

In [ ]:
train_data = pad_sequences(train_data, maxlen=20)
val_data = pad_sequences(val_data, maxlen=20)

In [ ]:
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])


history = model.fit(train_data, train_labels, epochs=10, verbose=1, batch_size=64, validation_data=(val_data, val_labels))

In [ ]:
model.summary()

### Evaluation plots

In [ ]:
loss = history.history['loss']
val_loss = history.history['val_loss']
epochs = range(1,len(loss)+1)
plt.plot(epochs,loss,'y',label='Training loss')
plt.plot(epochs,val_loss,'r',label='Validation loss')
plt.title('Training and validation loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()


acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
plt.plot(epochs,acc,'y',label='Training acc')
plt.plot(epochs,val_acc,'r',label='Validation acc')
plt.title('Training and validation accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.show()

### Model Testing

In [ ]:
test_loss, test_accuracy = model.evaluate(test_data, test_labels)
print("Test Loss:",np.round(test_loss,4))
print("Test Accuracy:", np.round(test_accuracy,4))

In [ ]:
test_data = pad_sequences(test_data, maxlen=20)

predictions = model.predict(test_data)

In [ ]:
predicted_labels = np.argmax(predictions, axis=1)

### Evaluation

In [ ]:
accuracy = accuracy_score(test_labels, predicted_labels)

In [ ]:
precision, recall, f1_score, _ = precision_recall_fscore_support(test_labels, predicted_labels, average="macro")

print("Accuracy:",np.round(accuracy,2))
print("Precision:", np.round(precision,2))
print("Recall:", np.round(recall,2))
print("F1-score:", np.round(f1_score,2))

In [ ]:
from sklearn.metrics import confusion_matrix
confusion = confusion_matrix(test_labels, predicted_labels)


plt.figure(figsize=(8, 6))
sns.heatmap(confusion, annot=True, fmt="d", cmap="Blues", cbar=False)

plt.title("Confusion Matrix")
plt.xlabel("Predicted Labels")
plt.ylabel("True Labels")
plt.xticks(ticks=[0, 1, 2], labels=["Negative", "Neutral", "Positive"])
plt.yticks(ticks=[0, 1, 2], labels=["Negative", "Neutral", "Positive"])

plt.show()